<a href="https://colab.research.google.com/github/hieu0978756924-spec/NLP/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch==2.2.2 \
transformers==4.41.0 \
tokenizers==0.19.1 \
huggingface-hub==0.36.2 \
accelerate==0.30.1 \
sentence-transformers==2.7.0 \
datasets==2.19.0 \
scikit-learn pandas numpy faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import transformers
import torch
import accelerate

print(transformers.__version__)
print(torch.__version__)
print(accelerate.__version__)
print(torch.cuda.is_available())

5.0.0
2.10.0+cu128
1.13.0
True


In [ ]:
import pandas as pd

du_lieu = pd.read_csv("medquad.csv")
du_lieu = du_lieu[['question','answer','focus_area']].dropna()

In [ ]:
# chuyển dữ liệu chữ thành số và đếm số nhóm trong cột focus_area
du_lieu['label'], labels = pd.factorize(du_lieu['focus_area'])
so_lop = len(labels)
print(so_lop)

5125


In [ ]:
from sklearn.model_selection import train_test_split
#train_test_split là àm của thư viện sklearn dùng để chia dữ liệu ra 2 phần
#Train set → dùng để huấn luyện mô hình
#Test set → dùng để kiểm tra độ chính xác mô hình
train_df, test_df = train_test_split(
    du_lieu, #toàn bộ dữ liệu ban đầu
    test_size=0.2,
    random_state=42 #kết quả chia dữ liệu luôn giống nhau mỗi lần chạy
)

In [ ]:
#tải mô hình BERT và tokenizer từ Hugging Face, chuẩn bị cho bài toán phân loại
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "bert-base-uncased" #uncased không phân biệt chữ hoa/thường
tokenizer = AutoTokenizer.from_pretrained(model_name)
#Tải BERT đã học sẵn,thêm lớp cuối để phân loại
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=so_lop
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import transformers
print(transformers.__file__)
print(transformers.__version__)

/usr/local/lib/python3.12/dist-packages/transformers/__init__.py
5.0.0


In [ ]:
import torch
#biến câu hỏi và đáp án thành dạng số để đưa vào AI học
class DatasetYTe(torch.utils.data.Dataset):
  #lấy dữ liệu vào
    def __init__(self, df):
        self.cau_hoi = df['question'].tolist()
        self.nhan = df['label'].tolist()
  # đếm dữ liệu
    def __len__(self):
        return len(self.cau_hoi)
# chạy khi AI lấy từng mẫu để học
    def __getitem__(self, idx):
      #biến câu thành số
        encoding = tokenizer(
            self.cau_hoi[idx],
          # chuẩn hóa độ dài câu ngắn -> thêm số 0 câu dài cắt bớt
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
      # làm gọn dữ liệu bỏ chiều thừa để dữ liệu gọn hơn
        item = {k: v.squeeze() for k, v in encoding.items()}
        #gắn nhãn đúng cho câu hỏi
        item['labels'] = torch.tensor(self.nhan[idx])
        return item
# tạo dataset
train_ds = DatasetYTe(train_df)
test_ds = DatasetYTe(test_df)

In [ ]:
from transformers import Trainer
print("OK")

OK


In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

du_lieu = pd.read_csv("medquad.csv")
du_lieu = du_lieu[['question','answer','focus_area']].dropna()

# chỉ lấy 100 loại bệnh phổ biến
TOP_N_CLASSES = 100
# lấy 100 nhóm bệnh xuất hiện nhiều nhất
top_focus_areas = du_lieu['focus_area'].value_counts().nlargest(TOP_N_CLASSES).index.tolist()
#Lọc dữ liệu giữ lại dl thuộc 100 nhóm đó
du_lieu_filtered = du_lieu[du_lieu['focus_area'].isin(top_focus_areas)].copy()

# tạo  labels số
du_lieu_filtered['label'], labels = pd.factorize(du_lieu_filtered['focus_area'])
# đếm có bao nhiêu loại bệnh
so_lop = len(labels)
print(f"Number of classes after filtering: {so_lop}")
# chia train /test
train_df, test_df = train_test_split(
    du_lieu_filtered,
    test_size=0.2,
    random_state=42,
    stratify=du_lieu_filtered['label']
)

#load BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
#biến bert thành máy phân loại bệnh
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=so_lop # Use updated so_lop
)
class DatasetYTe(torch.utils.data.Dataset):
    def __init__(self, df):
        self.cau_hoi = df['question'].tolist()
        self.nhan = df['label'].tolist()

    def __len__(self):
        return len(self.cau_hoi)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.cau_hoi[idx],
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        item = {k: v.squeeze() for k, v in encoding.items()}
        item['labels'] = torch.tensor(self.nhan[idx])
        return item

train_ds = DatasetYTe(train_df)
test_ds = DatasetYTe(test_df)

# cấu hình máy học
args = TrainingArguments(
    # lưu model sau khi train
    output_dir="./model_bert",
    #ttoocs độ học
    learning_rate=2e-5,
    # mỗi lần học 8 câu
    per_device_train_batch_size=8,
    #khi test cũng 8c/lần
    per_device_eval_batch_size=8,
    # học toannf bộ dữ liệu 3 lần
    num_train_epochs=3,
    #mỗi lần học xong 1 vòng → test 1 lần
    eval_strategy="epoch",
    #mỗi vòng học xong → lưu model 1 lần
    save_strategy="epoch",
    report_to="none"
)
# máy huấn luyện
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()

Number of classes after filtering: 100


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,3.881045
2,No log,3.255625
3,No log,3.033979


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=468, training_loss=3.791359143379407, metrics={'train_runtime': 143.0125, 'train_samples_per_second': 26.096, 'train_steps_per_second': 3.272, 'total_flos': 245698615406592.0, 'train_loss': 3.791359143379407, 'epoch': 3.0})

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
#model dự đoán(predict → lấy dự đoán)
pred = trainer.predict(test_ds)
# lấy kq dự đoán (argmax → chọn nhãn)
y_pred = np.argmax(pred.predictions, axis=1)
# lấy đáp án đúng
y_true = test_df['label'].values
# tính độ chính xác(accuracy → đo độ đúng)
print("Accuracy:", accuracy_score(y_true, y_pred))

Accuracy: 0.77491961414791


In [ ]:
import torch
# chọn máy để chạy
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
#danh sách nhãn
ten_benh = labels

def chatbot(cau_hoi):
    #Tokenize câu hỏi
    inputs = tokenizer(cau_hoi, return_tensors="pt", truncation=True, padding=True).to(device)
    # model dự đoán
    outputs = model(**inputs)
    #Lấy kết quả cao nhất
    label_id = torch.argmax(outputs.logits, dim=1).item()
    # đổi  số -> tên bệnh
    return ten_benh[label_id]

print(chatbot("What is glaucoma?"))

Glaucoma


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.2 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
# ======================
# 1. Hàm dự đoán bệnh (BERT)
# ======================
def du_doan_benh(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

    # map label -> tên bệnh
    benh = labels[pred]
    return benh

# 2. Hàm lấy giải thích (FAISS)

def lay_giai_thich(query, benh, k=10):
    query_vec = embedder.encode([query]).astype("float32")
    faiss.normalize_L2(query_vec)

    D, I = index.search(query_vec, k)

    ket_qua = []
    for i in I[0]:
        q = du_lieu.iloc[i]["question"]
        a = du_lieu.iloc[i]["answer"]
        focus = du_lieu.iloc[i]["focus_area"]

        # lọc bệnh khác
        if focus != benh:
            continue

        # lọc rác
        if len(a) < 100:
            continue
        if "@" in a or "www" in a:
            continue

        ket_qua.append(a)

    return ket_qua[:2]


# 3. PIPELINE CHÍNH
def chatbot_y_te(trieu_chung):
    # bước 1: đoán bệnh
    benh = du_doan_benh(trieu_chung)
    # bước 2: lấy giải thích
    giai_thich = lay_giai_thich(trieu_chung, benh)
    # format output
    print("\n=== KẾT QUẢ ===")
    print("Triệu chứng:", trieu_chung)
    print("👉 Bệnh dự đoán:", benh)

    print("\n👉 Giải thích:")
    for gt in giai_thich:
        print("\n-", gt)

# 4. TEST
chatbot_y_te("what is diabetes")


=== KẾT QUẢ ===
Triệu chứng: what is diabetes
👉 Bệnh dự đoán: Diabetes

👉 Giải thích:

- Diabetes is a disease in which your blood glucose, or blood sugar, levels are too high. Glucose comes from the foods you eat. Insulin is a hormone that helps the glucose get into your cells to give them energy. With type 1 diabetes, your body does not make insulin. With type 2 diabetes, the more common type, your body does not make or use insulin well. Without enough insulin, the glucose stays in your blood. You can also have prediabetes. This means that your blood sugar is higher than normal but not high enough to be called diabetes. Having prediabetes puts you at a higher risk of getting type 2 diabetes.    Over time, having too much glucose in your blood can cause serious problems. It can damage your eyes, kidneys, and nerves. Diabetes can also cause heart disease, stroke and even the need to remove a limb. Pregnant women can also get diabetes, called gestational diabetes.    Blood tests can sh